In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)
num_samples = 1000
base_signal = np.random.normal(0, 1, size=(num_samples, 1))
features = np.hstack([base_signal + np.random.normal(0, 0.05, size=(num_samples, 1)) for _ in range(8)])
true_weights = np.arange(1, features.shape[1] + 1).reshape(-1, 1) * 0.5
target = features.dot(true_weights).ravel() + np.random.normal(0, 0.5, size=num_samples)

Xtr, Xte, ytr, yte = train_test_split(features, target, test_size=0.2, random_state=42)
Xtr_mean = Xtr.mean(axis=0)
Xtr_std = Xtr.std(axis=0, ddof=0) + 1e-12
scaled_features_tr = (Xtr - Xtr_mean) / Xtr_std
scaled_features_te = (Xte - Xtr_mean) / Xtr_std

def add_intercept(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

Xtr_aug = add_intercept(scaled_features_tr)
Xte_aug = add_intercept(scaled_features_te)

def ridge_cost_and_grad(weights, X, y, lambda_val):
    n = X.shape[0]
    preds = X.dot(weights)
    errors = preds.ravel() - y.ravel()
    cost = (errors**2).sum() / (2*n) + (lambda_val / (2*n)) * (weights[1:]**2).sum()
    grad = (X.T.dot(errors.reshape(-1,1)) / n) + (lambda_val / n) * np.vstack([np.zeros((1,1)), weights[1:]])
    return cost, grad

def ridge_gradient_descent(X, y, learning_rate=0.01, lambda_val=0.0, num_iterations=2000):
    n_features = X.shape[1]
    weights = np.zeros((n_features, 1))
    loss_history = []
    for _ in range(num_iterations):
        cost, grad = ridge_cost_and_grad(weights, X, y, lambda_val)
        grad = np.clip(grad, -1e3, 1e3)          # gradient clipping
        weights -= learning_rate * grad
        if np.any(np.isnan(weights)) or np.any(np.isinf(weights)):
            break
        loss_history.append(cost)
    return weights, loss_history

learning_rate_values = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 0.1]
lambda_values = [1e-15, 1e-10, 1e-5, 1e-3, 0, 1, 10, 20]

best_combo = None
best_r2 = -np.inf
results = []

for lr in learning_rate_values:
    for lam in lambda_values:
        weights, loss_hist = ridge_gradient_descent(Xtr_aug, ytr.reshape(-1,1), learning_rate=lr, lambda_val=lam, num_iterations=3000)
        preds_test = Xte_aug.dot(weights).ravel()
        if np.any(np.isnan(preds_test)) or np.any(np.isinf(preds_test)):
            continue
        r2 = r2_score(yte, preds_test)
        mse = mean_squared_error(yte, preds_test)
        final_cost = loss_hist[-1] if len(loss_hist) > 0 else np.inf
        results.append({"learning_rate": lr, "lambda_val": lam, "r2": r2, "mse": mse, "final_cost": final_cost})
        if r2 > best_r2 and np.isfinite(r2):
            best_r2 = r2
            best_combo = {"learning_rate": lr, "lambda_val": lam, "weights": weights, "r2": r2, "mse": mse, "final_cost": final_cost}

print("Best combo (max R2):", best_combo)
results_df = pd.DataFrame(results).sort_values(by="r2", ascending=False).reset_index(drop=True)
print(results_df.head(10))


Best combo (max R2): {'learning_rate': 0.1, 'lambda_val': 1e-15, 'weights': array([[0.43222782],
       [1.57613405],
       [1.59861642],
       [1.60821991],
       [2.09070212],
       [2.17652617],
       [2.76857318],
       [2.85169483],
       [2.9947164 ]]), 'r2': 0.9992141402053286, 'mse': 0.2421237830025839, 'final_cost': np.float64(0.11635140857270883)}
   learning_rate    lambda_val        r2       mse  final_cost
0           0.10  0.000000e+00  0.999214  0.242124    0.116351
1           0.10  1.000000e-15  0.999214  0.242124    0.116351
2           0.10  1.000000e-10  0.999214  0.242124    0.116351
3           0.10  1.000000e-05  0.999214  0.242124    0.116352
4           0.10  1.000000e-03  0.999214  0.242124    0.116378
5           0.10  1.000000e+00  0.999212  0.242839    0.142723
6           0.05  1.000000e-15  0.999207  0.244437    0.119064
7           0.05  0.000000e+00  0.999207  0.244437    0.119064
8           0.05  1.000000e-10  0.999207  0.244437    0.119064
9  

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

# Try loading local CSV first; otherwise, make synthetic dataset
csv_path = "Hitters.csv"
if not csv_path or not os.path.exists(csv_path):
    np.random.seed(42)
    n = 322
    df = pd.DataFrame({
        "Years": np.random.randint(1, 25, n),
        "Hits": np.random.randint(20, 200, n),
        "Runs": np.random.randint(10, 100, n),
        "RBI": np.random.randint(5, 120, n),
        "Walks": np.random.randint(5, 80, n),
        "PutOuts": np.random.randint(100, 1500, n),
        "Assists": np.random.randint(0, 400, n),
        "Errors": np.random.randint(0, 30, n),
        "League": np.random.choice(["A", "N"], n),
        "Division": np.random.choice(["E", "W"], n),
        "NewLeague": np.random.choice(["A", "N"], n),
        "Salary": np.random.normal(500, 200, n).clip(min=100)
    })
    print("⚙️ Using generated synthetic dataset (Hitters-like).")
else:
    df = pd.read_csv(csv_path)
    print("📁 Loaded local Hitters.csv")

df = df.dropna(subset=["Salary"])
df = df.fillna(df.median(numeric_only=True))

cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

target_column = "Salary"
X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

alpha_value = 0.5748
lin_model = LinearRegression().fit(X_train_scaled, y_train)
ridge_model = Ridge(alpha=alpha_value).fit(X_train_scaled, y_train)
lasso_model = Lasso(alpha=alpha_value, max_iter=10000).fit(X_train_scaled, y_train)

def evaluate(model, name):
    preds = model.predict(X_test_scaled)
    print(f"{name}: R2={r2_score(y_test, preds):.4f}, RMSE={np.sqrt(mean_squared_error(y_test, preds)):.4f}")

evaluate(lin_model, "LinearRegression")
evaluate(ridge_model, "Ridge(alpha=0.5748)")
evaluate(lasso_model, "Lasso(alpha=0.5748)")


⚙️ Using generated synthetic dataset (Hitters-like).
LinearRegression: R2=-0.0459, RMSE=212.4371
Ridge(alpha=0.5748): R2=-0.0457, RMSE=212.4115
Lasso(alpha=0.5748): R2=-0.0416, RMSE=211.9954


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml

try:
    from sklearn.datasets import load_boston
    data = load_boston()
    X, y = data.data, data.target
except Exception:
    data = fetch_openml(name="boston", version=1, as_frame=True)
    X = data.data.values
    y = data.target.astype(float).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

alphas_to_try = np.logspace(-6, 6, 50)
ridge_cv = RidgeCV(alphas=alphas_to_try, cv=5).fit(X_train_scaled, y_train)
ridge_preds = ridge_cv.predict(X_test_scaled)
print(f"RidgeCV alpha={ridge_cv.alpha_:.6f} | R2={r2_score(y_test, ridge_preds):.4f} | RMSE={np.sqrt(mean_squared_error(y_test, ridge_preds)):.4f}")

lasso_cv = LassoCV(alphas=100, cv=5, max_iter=10000).fit(X_train_scaled, y_train)
lasso_preds = lasso_cv.predict(X_test_scaled)
print(f"LassoCV alpha={lasso_cv.alpha_:.6f} | R2={r2_score(y_test, lasso_preds):.4f} | RMSE={np.sqrt(mean_squared_error(y_test, lasso_preds)):.4f}")


RidgeCV alpha=2.329952 | R2=0.6681 | RMSE=4.9337
LassoCV alpha=0.006864 | R2=0.6684 | RMSE=4.9314


In [11]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

iris = load_iris()
features = iris.data
target = iris.target
class_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def add_intercept(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_ovr_logistic(X, y, learning_rate=0.1, num_iterations=5000, reg_strength=0.0):
    classes = np.unique(y)
    n_features = X.shape[1]
    weights_dict = {}
    for cls in classes:
        y_binary = (y == cls).astype(float).reshape(-1, 1)
        w = np.zeros((n_features, 1))
        for _ in range(num_iterations):
            preds = sigmoid(X.dot(w))
            error = preds - y_binary
            grad = (X.T.dot(error) / X.shape[0]) + (reg_strength / X.shape[0]) * np.vstack([np.zeros((1,1)), w[1:]])
            w -= learning_rate * grad
        weights_dict[int(cls)] = w
    return weights_dict

def predict_ovr(X, weights_dict):
    scores = np.hstack([X.dot(weights_dict[cls]) for cls in sorted(weights_dict.keys())])
    return np.argmax(scores, axis=1)

X_train_aug = add_intercept(X_train_scaled)
X_test_aug = add_intercept(X_test_scaled)
weights_dict = train_ovr_logistic(X_train_aug, y_train, learning_rate=0.5, num_iterations=3000, reg_strength=0.01)
y_pred = predict_ovr(X_test_aug, weights_dict)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9333333333333333
Classification report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Confusion matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
